# Lab 2.3 &mdash; Sub-goals That Finish, and Knowing When to Re-plan

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 1 &middot; Module 2 &mdash; Agentic Planning &amp; Reasoning**

### What you'll do
- Have the model return a <code>Plan</code> object with declared dependencies
- Write the field descriptions the model actually reads
- Tell a transient failure from a wrong plan &mdash; retry one, re-plan the other

> **How this lab works.** You write real LangChain and LangGraph code. Fill every `BLANK`,
> then run the **Self-check** cell under each section &mdash; those check the *objects you built*
> (a bound tool, a compiled graph, an emitted tool call), so they are deterministic and do not
> depend on the model. Cells marked **Run it for real** put your code in front of the sandbox
> model; that is the part worth watching. The score line is feedback, not a grade.

> **The thread.** All five Module 2 labs work one case: an internal employee help desk.
> The rules are ordinary on purpose &mdash; the only new thing here is how the agent reasons.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-2-03")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nSelf-check: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens: 24.1s / 980 tokens with it on, 0.7s / 29 with it off, for the same answer. Off is
# the default here because you will make a lot of calls today. Pass think=True to see the
# difference for yourself -- and note that prompts written as an explicit ordered procedure
# survive thinking being off, while vague ones do not.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

def show_messages(messages, width: int = 88) -> None:
    """Print a message list the way a trace reads: type, content, and any tool calls."""
    for m in messages:
        kind = getattr(m, "type", "?")
        body = str(getattr(m, "content", "")).replace("\n", " ")[:width]
        calls = getattr(m, "tool_calls", None)
        line = f"  [{kind:9}] {body}"
        if calls:
            line += "  -> calls: " + ", ".join(f"{c['name']}({c['args']})" for c in calls)
        print(line)

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------ the case file (synthetic, self-contained)
# An internal employee help desk. Ordinary rules on purpose: the only new thing in these five
# labs is LangChain. Nothing here is real data and nothing leaves this notebook.

REQUESTS = {
    "EHD-7001": {"who": "Priya Nair",   "category": "access",   "urgency": "high",
                 "wants": "reset",
                 "text": "Locked out of the payroll portal after the password reset."},
    "EHD-7002": {"who": "Rahul Menon",  "category": "hardware", "urgency": "high",
                 "wants": "replacement",
                 "text": "Laptop battery has swollen and the case is bulging."},
    "EHD-7003": {"who": "Anita Sharma", "category": "software", "urgency": "low",
                 "wants": "licence",
                 "text": "Need a licence for the diagramming tool, about 180 USD a year."},
    "EHD-7004": {"who": "Vikram Rao",   "category": "access",   "urgency": "medium",
                 "wants": "admin-rights",
                 "text": "Please give me admin rights on the finance reporting system."},
    "EHD-7005": {"who": "Priya Nair",   "category": "hardware", "urgency": "low",
                 "wants": "replacement",
                 "text": "Second monitor flickers every few minutes."},
}

# The handbook, one entry per category. Every judgement in this module comes from these.
HANDBOOK = {
    "access":   "Verify identity, then reset. The help desk NEVER grants elevated or admin "
                "rights -- route those to Identity and Access Management.",
    "hardware": "Replace under warranty. A swollen battery is a safety issue: stop use "
                "immediately and replace the same day, whatever urgency the employee set.",
    "software": "Licences over 100 USD per year need the cost-centre owner's approval first.",
}

SLA_HOURS = {"high": 4, "medium": 24, "low": 72}
ROUTE_OUT = {"admin-rights"}     # what the help desk must hand to another team, never do itself

print(f"{len(REQUESTS)} help desk requests, {len(HANDBOOK)} handbook entries loaded")

## Concept

EHD-7003 is a 180 USD licence, and the software handbook entry says licences over 100 USD need
the cost-centre owner's approval first. So the plan is not one step and it is not three
independent steps: **check the price &rarr; get approval &rarr; purchase**. Step three cannot start
until step two finishes, and step two only exists because of what step one found.

Two things have to be written down for that to work at all:

1. the plan has to be an **object**, with the order in it, not a paragraph you re-read every turn;
2. when a step fails you have to know **which kind of failure it was**, because a transient one and
   a wrong plan need opposite responses and the same `except` block catches both.

## Section 1 &mdash; A plan the model returns as an object

`with_structured_output(Plan)` sends your schema to the model and gives you back a `Plan`.

The `Field(description=...)` strings are the part everyone treats as documentation. They are not
documentation. LangChain sends them to the model **as the schema** &mdash; they are the only
instruction it ever gets about what belongs in each field.

In [ ]:
from typing import List
from pydantic import BaseModel, Field


class Step(BaseModel):
    """One step of a help desk plan."""
    action: str = Field(description="BLANK")
    # TODO ^ what must one step say? Something an analyst could carry out without asking you.

    depends_on: List[int] = Field(default_factory=list, description="BLANK")
    # TODO ^ these integers are what turns a list into an order. Say what they point at.

    done_when: str = Field(
        description="A test somebody else could apply to say this step is finished")


class Plan(BaseModel):
    """An ordered plan for handling one help desk request."""
    goal: str = Field(description="The outcome this plan reaches, in one line")
    steps: List[Step] = Field(description="The steps, in the order they should run")

In [ ]:
# --- Self-check: Section 1   (the schema object, before any model sees it)
def described(field: str) -> str:
    """The description the model will be sent. An unfilled blank is the literal 'BLANK'."""
    d = Step.model_fields[field].description
    if d.strip() == "BLANK":
        raise NameError(f"Step.{field} description is still BLANK")   # -> [TODO], not [FAIL]
    return d


def _accepts(step: dict) -> bool:
    """Does Plan let this step through?"""
    try:
        Plan(goal="g", steps=[step])
        return True
    except NameError:
        raise                       # a blank above, not a schema verdict
    except Exception:
        return False


def ehd_7003_plan() -> Plan:
    """The three steps the software handbook forces for a 180 USD licence."""
    return Plan(goal="Anita Sharma has a licence for the diagramming tool, or a written refusal",
                steps=[Step(action="check the annual price of the licence",
                            depends_on=[], done_when="the price in USD is written down"),
                       Step(action="get the cost-centre owner's approval",
                            depends_on=[1], done_when="the owner has replied yes or no"),
                       Step(action="purchase the licence and send the key",
                            depends_on=[2], done_when="Anita has the key")])


check("the action description tells the model to write one concrete thing to do",
      lambda: len(described("action")) > 30)
check("the depends_on description says what the numbers point at",
      lambda: "step" in described("depends_on").lower() and len(described("depends_on")) > 30,
      "a bare list of integers means nothing to the model unless you say what they index")
check("Plan can express the EHD-7003 dependency: approval before purchase",
      lambda: ehd_7003_plan().steps[2].depends_on == [2]
              and ehd_7003_plan().steps[0].depends_on == [])
check("Plan rejects a step with no completion test",
      lambda: not _accepts({"action": "buy it", "depends_on": []})
              and _accepts({"action": "buy it", "depends_on": [], "done_when": "key sent"}),
      "a sub-goal no one can call finished is how an agent loops forever on step three")
score()

## Section 2 &mdash; Transient failure, or wrong plan

The licence catalogue returns `503`. Retry it &mdash; bounded, because an unbounded retry is a hang
with a progress bar.

The catalogue returns `no such tool: grant_admin_rights`. Retrying that is not optimism, it is
arithmetic: the identical call will fail identically forever. What is wrong is the **plan**, and
the only move that changes anything is to make a new one.

The commonest version of this bug in production is a single blanket `except: retry`.

In [ ]:
TRANSIENT = ("503", "502", "timeout", "timed out", "rate limit", "connection reset")
WRONG_PLAN = ("no such tool", "no such request", "invalid argument", "unknown field",
              "not permitted for this desk")

RETRY_BUDGET = 2


def classify_failure(error: str) -> str:
    """'retry' -- the world misbehaved.  're-plan' -- the plan was wrong."""
    e = error.lower()
    world_misbehaved = any(s in e for s in TRANSIENT)
    plan_was_wrong = any(s in e for s in WRONG_PLAN)

    if plan_was_wrong:
        return BLANK        # TODO: this exact call, sent again, gets this exact error. So?
    if world_misbehaved:
        return "retry"
    return "re-plan"        # unrecognised: assume the plan, because that is the cheaper mistake


def next_move(error: str, attempts_so_far: int) -> str:
    """What to do after a failed step: 'retry', 're-plan' or 'give up'."""
    verdict = classify_failure(error)
    if verdict == "retry" and attempts_so_far >= RETRY_BUDGET:
        return "give up"    # a bounded retry is a retry; an unbounded one is an outage
    return verdict

In [ ]:
# --- Self-check: Section 2   (hand-written errors, no model, no network)
FAILURES = {
    "503 Service Unavailable from the licence catalogue": "retry",
    "rate limit exceeded on the approvals API, try again in 30s": "retry",
    "no such tool: grant_admin_rights": "re-plan",
    "no such request: EHD-9999": "re-plan",
    "invalid argument: amount must be a number, got '180 USD'": "re-plan",
}

check("every transient failure is retried",
      lambda: all(classify_failure(e) == "retry" for e, v in FAILURES.items() if v == "retry"))
check("a wrong-plan failure is never retried",
      lambda: all(classify_failure(e) == "re-plan" for e, v in FAILURES.items() if v == "re-plan"),
      "'no such tool' is not a blip -- the same call will fail identically every time")
check("an unrecognised failure re-plans rather than hammering",
      lambda: classify_failure("the printer is on fire") == "re-plan")
check("the retry is bounded",
      lambda: next_move("503 Service Unavailable", 0) == "retry"
              and next_move("503 Service Unavailable", RETRY_BUDGET) == "give up")
score()

## Run it for real

Ask the model for a `Plan` for EHD-7003, then put the two kinds of failure through the classifier.

In [ ]:
def real_plan(rid: str) -> Plan:
    r = REQUESTS[rid]
    return get_llm().with_structured_output(Plan).invoke(
        "You are an employee help desk analyst. Plan the handling of this request.\n"
        f"REQUEST: {json.dumps({'id': rid, **r})}\n"
        f"HANDBOOK ({r['category']}): {HANDBOOK[r['category']]}\n"
        "Use depends_on to say which steps cannot start until an earlier one has finished.")


if llm_ready():
    plan = guard(lambda: real_plan("EHD-7003"))
    if plan:
        print("goal:", plan.goal, "\n")
        for i, s in enumerate(plan.steps, 1):
            print(f"{i}. {s.action}")
            print(f"   after step(s) {s.depends_on or 'none'} | done when: {s.done_when}")

In [ ]:
def show_recovery():
    for err, attempts in [("503 Service Unavailable from the licence catalogue", 0),
                          ("503 Service Unavailable from the licence catalogue", RETRY_BUDGET),
                          ("no such tool: grant_admin_rights", 0)]:
        print(f"{next_move(err, attempts):8}  <- attempt {attempts}: {err}")

guard(show_recovery)

### Read it

The model returned a `Plan`, not a paragraph. Look at what that bought: `depends_on` is a list of
integers you can sort on, so "may this step start yet?" is a comparison rather than a re-read of
prose. If the model put purchase before approval, you can see that without the model's help.

The two failures went in looking almost identical &mdash; both are strings from a service that said
no &mdash; and came out with opposite moves. The 503 is worth two more attempts and then a stop.
`no such tool: grant_admin_rights` is worth zero attempts, because nothing about attempt two is
different from attempt one.

Note the default in the third branch. Unrecognised failures re-plan. Guessing "transient" costs you
a retry loop; guessing "wrong plan" costs you one extra planning call.

In [ ]:
score()

## Your turn

1. Delete the `done_when` field from `Step` and ask the model for the EHD-7003 plan again. Read
   step three. Could an agent tell whether it had finished?
2. Move `"invalid argument"` from `WRONG_PLAN` to `TRANSIENT` and re-run the Section 2 check. Count
   the calls a real agent would then make against `amount='180 USD'`.